In [ ]:
from emu_renewal.outputs import get_output_dir, get_combined_df, get_df_from_3darray
import arviz as az
import pandas as pd
import numpy as np
from emu_renewal.plotting import plot_proc_comparison
import seaborn as sns

In [ ]:
country = "France"
no_mob_time = "15122024_1429"
mob_time = "15122024_1418"

In [ ]:
no_mob_idata = az.from_netcdf(get_output_dir(country, "no_mob", no_mob_time) / "idata.nc")
mob_idata = az.from_netcdf(get_output_dir(country, "mob", mob_time) / "idata.nc")

In [ ]:
plot_proc_comparison(no_mob_idata, mob_idata, ["without mobility", "with mobility"])

In [ ]:
# Get the variable process and dispersion values, with samples as first axis (for use as index to dataframe later)
mob_proc_vals = np.swapaxes(mob_idata.posterior["proc"].to_numpy(), 0, 1)
mob_disp_vals = np.swapaxes(mob_idata.posterior["dispersion_proc"].to_numpy(), 0, 1)
no_mob_proc_vals = np.swapaxes(no_mob_idata.posterior["proc"].to_numpy(), 0, 1)
no_mob_disp_vals = np.swapaxes(no_mob_idata.posterior["dispersion_proc"].to_numpy(), 0, 1)

# Get dataframes from arrays
mob_proc_df = get_df_from_3darray(mob_proc_vals, [0, 2, 1])
no_mob_proc_df = get_df_from_3darray(no_mob_proc_vals, [0, 2, 1])

# Combine the mobility and no mobility dataframes
proc_comparison_df = get_combined_df(mob_proc_df, no_mob_proc_df, "mob", "no_mob")

# Combine the dispersion values into a dataframe
disp_comparison_df = pd.DataFrame(
    {
        "mob": mob_disp_vals.flatten(),
        "no_mob": no_mob_disp_vals.flatten(),
    }
)

In [ ]:
sns.kdeplot(disp_comparison_df, fill=True)

In [ ]:
sns.kdeplot(proc_comparison_df, fill=True)